# 文をベクトルにして意味の近さを測る (Sentence Transformers)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/10_similarity_search/sentence_embeddings.ipynb)

NLP クックブック 10-01 ／ 文埋め込み・類似度・検索

言い回しが違っても意味が近い文を見つけたい。文を埋め込みベクトルにしてコサイン類似度を計算する。

In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-small")

sentences = ["今日は天気が良いので散歩に行きます。", "晴れているから外を歩いてくるよ。", "明日の会議は午後 3 時からです。",
             "The weather is nice, so I'll go for a walk.", "機械学習モデルの学習には GPU が必要だ。"]
# E5 系のモデルは入力の先頭に "query: " か "passage: " を付ける約束になっている
emb = model.encode(["query: " + s for s in sentences], normalize_embeddings=True)
print(emb.shape)

(5, 384)


In [2]:
import pandas as pd

sim = model.similarity(emb, emb)  # コサイン類似度の行列
pd.DataFrame(sim.numpy(), index=[s[:12] for s in sentences], columns=range(len(sentences))).round(2)

,0,1,2,3,4
今日は天気が良いので散歩,1.00,0.91,0.84,0.85,0.83
晴れているから外を歩いて,0.91,1.00,0.82,0.81,0.80
明日の会議は午後 3 時,0.84,0.82,1.00,0.73,0.81
The weather,0.85,0.81,0.73,1.00,0.72
機械学習モデルの学習には,0.83,0.80,0.81,0.72,1.00


- 日本語と英語の同じ意味の文も近くなる (多言語モデル)。
- モデルごとにプレフィックスの約束が違う (E5: `query:` / `passage:`、Ruri: `検索クエリ:` / `検索文書:` など)。モデルカードを必ず確認する。
- 日本語特化なら `cl-nagoya/ruri-v3-310m` や `sbintuitions/sarashina-embedding-v2-1b` など。[JMTEB](https://github.com/sbintuitions/JMTEB) のリーダーボードが参考になる。